# ハンズオン0: 量子コンピュータって、何をしているの？

量子コンピュータの「いちばん最初の一歩」を、手を動かしながら体験します。

## このハンズオンが終わったら

次の3つを、自分の言葉で誰かに説明できる状態を目指します。

1. 量子ビットは「測るまで、0か1かが決まっていない」ものだということ
2. 量子コンピュータの答えは1回では分からず、何度も試して確かめるということ
3. 2つの量子ビットが「必ず連動する」という、普通のコインでは起こせない現象があること

## 数学はどのくらい必要？

平方根（√）と、割り算で出てくる確率くらいです。式が出てきたら、すぐ下に日本語の言い換えを必ず書いてあります。式が苦手な方は、言い換えのほうだけ読んでも先に進めます。

## 本物の量子コンピュータを使うの？

いいえ。このノートブックは、普通のパソコンの中で量子コンピュータの「ふるまいを真似」するだけです。料金は一切かからないので、何度でも実行してかまいません。

最後に、本物のAzure Quantumにつなぐ方法も紹介しますが、そこは最初はオフにしてあります。

## まずは準備

次のセルは準備運動です。計算に使う道具（ライブラリ）を読み込み、`SEED`（乱数の種）を固定します。種を固定すると、何度実行しても同じ結果になるので、教材として結果を見比べやすくなります。

In [ ]:
import cmath
import math
import random
from collections import Counter

SEED = 42
random.seed(SEED)

## 1. 量子ビットは「どっちつかず」でいられる

### 普通のコンピュータの場合

私たちが使っているパソコンは、情報を**ビット**で持ちます。ビットは必ず `0` か `1` のどちらかです。電気が流れていないか、流れているか。その2択しかありません。

### 量子ビットの場合

量子コンピュータが使う**量子ビット**は、測定する前なら「0の可能性」と「1の可能性」を**同時に持つ**ことができます。これを**重ね合わせ**と呼びます。

### 身近なたとえ

机の上で回っているコインを想像してください。回っている最中は「表」とも「裏」とも言えません。手で押さえた瞬間に、どちらかに決まります。量子ビットも、測った瞬間に `0` か `1` に決まります。

ただしこのたとえは完璧ではありません。回るコインは「本当はもう決まっているが、私たちが知らないだけ」です。一方の量子ビットは「本当にまだ決まっていない」ので、可能性どうしが**打ち消し合う**ことまで起こります。この打ち消し合いこそ、量子コンピュータの計算の心臓部です。

### 「振幅」という数字

それぞれの可能性には**振幅**という数字がついています。関係はとてもシンプルです。

> **振幅を2乗すると、その結果が出る確率になる**

この「2乗する」がポイントです。振幅はマイナスにもなれるので、プラスの振幅とマイナスの振幅が出会うと打ち消し合えます。確率そのものはマイナスになれないので、こういう芸当はできません。

### Hadamard（アダマール）ゲート

**ゲート**とは、量子ビットに対する操作のことです。なかでもHadamardゲートは「`0` の状態を、`0` と `1` がちょうど半々の状態にする」操作で、量子プログラムの出発点として頻繁に登場します。

半々の状態の振幅は、どちらも `1 / √2`（およそ 0.707）です。2乗すると 0.5、つまり50%になります。

### 次のセルがすること

1. 半々の状態を表す2つの振幅を用意する
2. それぞれを2乗して、確率に直す
3. 確率の合計が `1`（＝100%）になっていることを確かめる

`abs(amplitude) ** 2` が「振幅を2乗して確率にする」部分です。最後の `assert` は「もし違っていたら、そこで止めて教えて」という確認用の命令で、テストの役割をしています。

In [ ]:
sqrt2 = math.sqrt(2)
state = [1 / sqrt2, 1 / sqrt2]
probabilities = [abs(amplitude) ** 2 for amplitude in state]

print("state amplitudes:", state)
print("probabilities:", probabilities)
assert math.isclose(sum(probabilities), 1.0)

## 2. 1回測っただけでは、何も分からない

### 測ると、重ね合わせは壊れる

量子ビットを測定すると、「半々」だった状態は壊れて、`0` か `1` のどちらかに確定します。しかも結果は毎回変わります。

### サイコロのたとえ

サイコロを1回振って `3` が出たとします。これだけでは「1が出る確率は6分の1」とは分かりません。60回振って1が10回くらい出る、600回振って100回くらい出る、と繰り返してはじめて確率が見えてきます。

量子コンピュータもまったく同じです。**同じプログラムを何度も実行して、結果を数える**必要があります。

### shots（ショット数）

この「何回実行するか」を**shots**と呼びます。量子コンピュータを使うときに必ず指定する、最も基本的な設定です。

- shotsが少ない → 結果はブレる。ただし速くて安い
- shotsが多い → 結果は正確になる。ただし時間もお金もかかる

本物の機械は実行回数で課金されることが多いので、shotsはそのまま費用に直結します。

### 次のセルがすること

「1が出る確率が50%のコイン」をshots回投げて、`0` と `1` が何回出たか数えます。shotsを10回、100回、10000回と増やしたときに、結果の割合が理論値の `0.500` にどう近づくかを見てください。

10回だけだと `0.600` のように大きくズレることがあります。これは計算が間違っているのではなく、**回数が少ないとそういうもの**です。

In [ ]:
def sample_qubit(probability_one, shots, seed):
    rng = random.Random(seed)
    return Counter("1" if rng.random() < probability_one else "0" for _ in range(shots))

for shots in (10, 100, 10_000):
    counts = sample_qubit(probabilities[1], shots, SEED)
    print(f"shots={shots:>5}: {dict(counts)}, P(1)={counts['1'] / shots:.3f}")

## 3. 2つの量子ビットが「必ず連動する」（もつれ）

ここからが、普通のコンピュータでは絶対に真似できない現象です。

### 不思議な手品のたとえ

2枚のコインを用意して、1枚を東京、もう1枚を大阪に持っていきます。同時に投げます。

- 東京のコインは、表と裏が半々でバラバラに出る
- 大阪のコインも、表と裏が半々でバラバラに出る
- なのに、あとで答え合わせをすると**必ず同じ面**が出ている

普通のコインでは、こんなことは起こりません。しかし量子ビットでは実際に起こります。これを**もつれ**（エンタングルメント）と呼びます。

### Bell（ベル）状態

いちばん有名なもつれの形が**Bell状態**です。2つの量子ビットを測ると、

- `00`（両方0）が出る確率 … 50%
- `11`（両方1）が出る確率 … 50%
- `01` や `10`（バラバラ）が出る確率 … **0%**

となります。片方だけを見ればランダムなのに、2つ並べると必ず一致する。この「個別にはランダムなのに、全体では強く結びついている」性質が、量子コンピュータの計算力を支える柱のひとつです。

### 次の2つのセルがすること

- 1つ目: 4パターンそれぞれの確率を計算し、`01` と `10` が本当に `0` であることを確かめます
- 2つ目: 実際に1000回測ってみて、**毎回必ず2つの結果が一致する**ことを確かめます

`Counter` は「どの結果が何回出たか」を数えてくれる道具です。

In [ ]:
bell_state = {
    "00": 1 / sqrt2,
    "01": 0j,
    "10": 0j,
    "11": 1 / sqrt2,
}
bell_probabilities = {bits: abs(amplitude) ** 2 for bits, amplitude in bell_state.items()}
print(bell_probabilities)
assert math.isclose(sum(bell_probabilities.values()), 1.0)
assert bell_probabilities["01"] == bell_probabilities["10"] == 0

In [ ]:
def sample_bell(shots, seed):
    rng = random.Random(seed)
    return Counter("00" if rng.random() < 0.5 else "11" for _ in range(shots))

counts = sample_bell(1_000, SEED)
print(counts)
correlation = sum(count for bits, count in counts.items() if bits[0] == bits[1]) / sum(counts.values())
print(f"同じ結果になる割合: {correlation:.3f}")
assert correlation == 1.0

## 4. 同じことを、量子コンピュータの言葉で書くと

ここまではPythonで「真似」をしてきました。本物の量子コンピュータに指示を出すときは、**Q#**（キューシャープ）のような専用の言語を使います。Microsoftが量子プログラミングのために作った言語です。

### コードの読み方

上のQ#コードは、さきほどのBell状態を作る手順そのものです。1行ずつ日本語にすると、こうなります。

| コード | 意味 |
|---|---|
| `use (q0, q1) = (Qubit(), Qubit());` | 量子ビットを2本借りる |
| `H(q0);` | 1本目に、半々にするHadamardゲートをかける |
| `CNOT(q0, q1);` | 1本目の状態に2本目を連動させる（ここでもつれる） |
| `MResetZ(q0)` | 測って結果を読み、量子ビットをきれいに戻して返す |

たった3行の操作で、さきほどの「不思議な手品」が作れます。量子プログラムが意外と短いことが分かると思います。

### 本物の機械で動かすと、どうなる？

理論上は `00` と `11` しか出ないはずです。ところが実機では `01` や `10` も数パーセント出てきます。

これはプログラムの間違いではありません。現在の量子コンピュータは非常に繊細で、わずかな温度変化や電気的な乱れで誤差が生じます。これを**ノイズ**と呼びます。

ノイズがあるからこそ、実験したときは次を必ず記録してください。あとで結果を正しく解釈するために不可欠です。

- 使った機械の名前
- shots（実行回数）
- 実行した日時
- 得られた結果の内訳

## 5. Azure Quantumにつないでみる（任意・料金が発生する場合あり）

ここまでは、すべて自分のパソコンの中だけで完結していました。ここから先は、クラウド上にある**本物の機械や高性能シミュレーター**につなぐ話です。

### 安全のため、最初はオフになっています

次のセルは `RUN_AZURE = False` となっていて、そのまま実行しても何も送信されません。料金も発生しません。試すときだけ `True` に変えてください。

### 試す前に必要なもの

1. Azureのアカウントと、Azure Quantumのワークスペース（講義本文の第4章で作り方を説明しています）
2. ワークスペースの「Resource ID」を、環境変数 `AZURE_QUANTUM_RESOURCE_ID` に設定しておくこと
3. どの機械を使うか、いくらかかるかの確認

### なぜ環境変数を使うの？

IDやパスワードをノートブックに直接書いてしまうと、そのファイルを誰かに渡したときに、中身も一緒に渡ってしまいます。GitHubに公開すれば全世界に見えます。

**接続情報はコードの外に置く。** これはセキュリティの基本中の基本です。

In [ ]:
RUN_AZURE = False

if RUN_AZURE:
    import os
    from qdk.azure import Workspace

    resource_id = os.environ["AZURE_QUANTUM_RESOURCE_ID"]
    workspace = Workspace(resource_id=resource_id)
    targets = workspace.get_targets()
    for target in targets:
        print(target.name)
else:
    print("Azure接続は無効です。ローカル演習だけを実行しました。")

### ジョブを送る流れ（雰囲気がつかめればOK）

実際にプログラムを送るときは、だいたいこの順番になります。

1. どの機械向けに作るかを決める
2. プログラムを、機械が読める形に翻訳する（コンパイル）
3. 今使える機械の一覧から、相性の良いものを選ぶ
4. shots（実行回数）を指定して送信する
5. **順番待ちの列に並び**、自分の番が来たら実行され、結果が返ってくる

### 普通のプログラムとの大きな違い

4番と5番です。世界中に数台しかない機械を、たくさんの人が共有しています。そのため「送ってすぐ結果が返る」とは限らず、混み具合によっては待つことがあります。

この待ち時間も含めて「実際に使えるか」を考えるのが、実務ではとても大切になります。

公式の手順はこちら: https://learn.microsoft.com/azure/quantum/how-to-submit-jobs-python

## 確認課題

答えを見ずに、まず自分で考えてみてください。

### 問題1

上のセルで shots を `100` と `10000` に変えて実行し、結果が `0.500` からどれだけズレるか比べてください。

**考えるポイント:** なぜ回数を増やすとズレが小さくなるのでしょうか。サイコロを何回も振る話を思い出してください。

### 問題2

Bell状態では、片方の量子ビットだけを見ると50%ずつランダムなのに、2つを並べると必ず一致します。これはなぜ矛盾ではないのか、友人に説明するつもりで書いてみてください。

**ヒント:** 「それぞれが何になるか」と「2つの関係がどうなっているか」は別の情報です。

### 問題3

本物の機械で実行したら、`01` が3%出てしまいました。上司に報告するとき、結果の数字以外に何を書き添えるべきでしょうか。5つ挙げてください。

**ヒント:** 他の人が同じ実験を再現できるために必要な情報は何でしょうか。